# Scratch seq2seq — inference runs only

Greedy-decode headings from **`best.pt`** checkpoints and write **one JSON per model size** under **`artifacts/inference_runs/`** (`{dev|test}_scratch_{tiny|medium|small}.json`).

**Workflow:** (1) **Setup** → **Load eval rows + helper** (materializes rows and defines `run_inference_for_checkpoint`) → run **Tiny** / **Small** / **Medium** cells as needed → optional **Saved runs**. Then **[`07_automated_metrics.ipynb`](07_automated_metrics.ipynb)** / **[`08_llm_judge.ipynb`](08_llm_judge.ipynb)** — those **only import** JSON (no checkpoint load).

**Requirements:** `HUGGINGFACE_HUB_TOKEN` (or `HF_TOKEN`) in `.env`.

**Eval scope:** **`ROW_LIMIT`** below — `None` loads **every row** in the Hub split (`dev` or `test`); an integer caps how many **rows** you decode for a faster run.

**Terminology:** A **row** is one dataset item (`source` passage + gold `target`). This notebook logs a **short source preview** (first **`PREVIEW_SOURCE_TOKENS`** tokenizer tokens only) plus the **full greedy prediction string** for each row. The JSON artifact still stores **complete** `sources`, `refs`, and `preds` for every row so **`07` / `08`** keep working.

Match **`EVAL_N`** in **`08_llm_judge.ipynb`** / **`07_automated_metrics.ipynb`** to the saved **`eval_n`** (printed below), or expect a mismatch warning.

**Progress:** during each checkpoint’s greedy pass, **`INFERENCE_LOG_EVERY`** (e.g. 100) prints **wall-clock** and **rows/s** to **stderr** every *N* rows, then a final **`complete:`** line. Set to **`0`** to turn that off.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "src" / "briefme").is_dir():
        return cwd
    if (cwd.parent / "src" / "briefme").is_dir():
        return cwd.parent
    return cwd


REPO_ROOT = _repo_root()
SRC_ROOT = REPO_ROOT / "src"

try:
    import briefme as _briefme_check  # noqa: F401
except ImportError:
    if SRC_ROOT.is_dir():
        sys.path.insert(0, str(SRC_ROOT))

load_dotenv(REPO_ROOT / ".env", override=True)

from briefme.data import SPLIT_DEV, SPLIT_TEST
from briefme.inference_persist import inference_runs_dir, save_scratch_inference_json
from briefme.metrics import aggregate
from briefme.seq2seq_data import materialize_examples
from briefme.scratch_inference import greedy_predict_headings, load_scratch_seq2seq_checkpoint

print("Repo:", REPO_ROOT)

Repo: /Users/naataaniitsosie/repos/cs474


### Load eval rows + helper

Materializes **`rows`** / **`refs`** / **`sources_full`** and defines **`run_inference_for_checkpoint(label, ckpt_path)`**. Re-run this section if you change **`EVAL_SPLIT`**, **`ROW_LIMIT`**, or knobs shared across checkpoints.

In [2]:
def _preview_first_tokens(text: str, tokenizer, *, max_tokens: int) -> tuple[str, bool]:
    """Return (decoded preview of first ``max_tokens`` ids, whether text extends past that)."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    clipped = len(ids) > max_tokens
    chunk = ids[:max_tokens]
    return tokenizer.decode(chunk, skip_special_tokens=True).strip(), clipped


# --- knobs (align EVAL_N in 02 / 03 with eval_n printed below) ---
EVAL_SPLIT = SPLIT_DEV  # or SPLIT_TEST
ROW_LIMIT = None  # None = all rows in split; or e.g. 30 for a fast subset
PREVIEW_SOURCE_TOKENS = 24  # logged source preview only; JSON still has full source text
LOG_MAX_ROWS = None  # None = log every row's preview + full pred; set an int to cap stdout
PRED_BATCH = 8
INF_DEVICE = "mps"  # "cuda" / "mps" / "cpu"
INFERENCE_LOG_EVERY = 100  # progress on stderr every N rows; 0 = quiet during decode

CKPT_TINY = REPO_ROOT / "runs" / "notebook_scratch_tiny_full" / "best.pt"
CKPT_SMALL = REPO_ROOT / "runs" / "notebook_scratch_small_full" / "best.pt"
CKPT_MEDIUM = REPO_ROOT / "runs" / "notebook_scratch_medium_full" / "best.pt"

split_tag = "dev" if EVAL_SPLIT == SPLIT_DEV else "test"
INFERENCE_DIR = inference_runs_dir(REPO_ROOT)

if not (os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN")):
    raise RuntimeError(
        "Set HUGGINGFACE_HUB_TOKEN (or HF_TOKEN) in .env to load splits from the Hub."
    )

rows = materialize_examples(EVAL_SPLIT, limit=ROW_LIMIT)
eval_n = len(rows)
refs = [r["target"] for r in rows]
sources_full = [r["source"] for r in rows]
lim_note = "all rows" if ROW_LIMIT is None else str(ROW_LIMIT)
print(f"[briefme] {EVAL_SPLIT!r}: {eval_n} rows (ROW_LIMIT={lim_note})", flush=True)


def run_inference_for_checkpoint(label: str, ckpt_path: Path) -> None:
    """Greedy-decode, save JSON, print corpus metrics + optional per-row preview."""
    if not ckpt_path.is_file():
        print(f"[skip] no checkpoint at {ckpt_path}")
        return

    print(f"\n=== {split_tag} · scratch {label} ({ckpt_path}) ===")
    model, tokenizer, tcfg, src_prefix = load_scratch_seq2seq_checkpoint(ckpt_path, device=INF_DEVICE)
    preds = greedy_predict_headings(
        model,
        tokenizer,
        rows,
        tcfg=tcfg,
        source_prefix=src_prefix,
        batch_size=PRED_BATCH,
        log_every=INFERENCE_LOG_EVERY,
        log_label=f"{split_tag}-{label}",
    )
    agg = aggregate(preds, refs)
    save_path = INFERENCE_DIR / f"{split_tag}_scratch_{label}.json"
    save_scratch_inference_json(
        save_path,
        split_tag=split_tag,
        eval_n=eval_n,
        label=label,
        checkpoint=ckpt_path,
        preds=preds,
        refs=refs,
        sources=sources_full,
        agg=agg,
    )
    print(f"Saved → {save_path.relative_to(REPO_ROOT)}  eval_n={eval_n}")
    print(
        "corpus:",
        {k: round(float(agg[k]), 4) for k in ("rouge1_f", "rougeL_f", "token_f1_macro", "chrf_corpus")},
    )

    n_log = eval_n if LOG_MAX_ROWS is None else min(LOG_MAX_ROWS, eval_n)
    print(f"\nPer-row log (source ≈ first {PREVIEW_SOURCE_TOKENS} tok; pred = full string):", flush=True)
    for i in range(n_log):
        pv, clipped = _preview_first_tokens(sources_full[i], tokenizer, max_tokens=PREVIEW_SOURCE_TOKENS)
        suffix = " …" if clipped else ""
        print(f"  [{i}] src≈ {pv}{suffix}")
        print(f"       pred: {preds[i]}")
    if LOG_MAX_ROWS is not None and eval_n > n_log:
        print(f"  … ({eval_n - n_log} rows omitted; set LOG_MAX_ROWS=None to print all)", flush=True)

    del model


[briefme] streaming 'dev' from Hub (limit=full split)...
[briefme]   'dev': 500 rows...
[briefme]   'dev': 1000 rows...
[briefme]   'dev': 1500 rows...
[briefme]   'dev': 2000 rows...
[briefme]   'dev': done (2319 rows)


[briefme] 'dev': 2319 rows (ROW_LIMIT=all rows)


### Tiny preset

Runs **`notebook_scratch_tiny_full/best.pt`**. Run **Load eval rows + helper** above first.


In [3]:
run_inference_for_checkpoint("tiny", CKPT_TINY)



=== dev · scratch tiny (/Users/naataaniitsosie/repos/cs474/runs/notebook_scratch_tiny_full/best.pt) ===


[dev-tiny] 100/2319 rows  8.5s wall  (~11.70 rows/s)
[dev-tiny] 200/2319 rows  15.5s wall  (~12.93 rows/s)
[dev-tiny] 300/2319 rows  23.4s wall  (~12.82 rows/s)
[dev-tiny] 400/2319 rows  30.7s wall  (~13.03 rows/s)
[dev-tiny] 500/2319 rows  38.6s wall  (~12.96 rows/s)
[dev-tiny] 600/2319 rows  46.4s wall  (~12.93 rows/s)
[dev-tiny] 700/2319 rows  54.9s wall  (~12.76 rows/s)
[dev-tiny] 800/2319 rows  62.4s wall  (~12.83 rows/s)
[dev-tiny] 900/2319 rows  70.4s wall  (~12.78 rows/s)
[dev-tiny] 1000/2319 rows  78.1s wall  (~12.80 rows/s)
[dev-tiny] 1100/2319 rows  86.4s wall  (~12.73 rows/s)
[dev-tiny] 1200/2319 rows  94.1s wall  (~12.75 rows/s)
[dev-tiny] 1300/2319 rows  102.3s wall  (~12.71 rows/s)
[dev-tiny] 1400/2319 rows  109.9s wall  (~12.74 rows/s)
[dev-tiny] 1500/2319 rows  117.8s wall  (~12.73 rows/s)
[dev-tiny] 1600/2319 rows  125.4s wall  (~12.76 rows/s)
[dev-tiny] 1700/2319 rows  134.0s wall  (~12.69 rows/s)
[dev-tiny] 1800/2319 rows  141.7s wall  (~12.70 rows/s)
[dev-tiny] 190

Saved → artifacts/inference_runs/dev_scratch_tiny.json  eval_n=2319
corpus: {'rouge1_f': 0.1125, 'rougeL_f': 0.0988, 'token_f1_macro': 0.0967, 'chrf_corpus': 0.3689}

Per-row log (source ≈ first 24 tok; pred = full string):


Token indices sequence length is longer than the specified maximum sequence length for this model (1132 > 512). Running this sequence through the model will result in indexing errors


  [0] src≈ Safety The Second Amendment states : "A well regulated Militia, being necessary to the security of …
       pred: The Court's text of the a "rement Clauses a "rement"s"
  [1] src≈ a. Firea rms serve valuable purposes, including ena- bling self - …
       pred: The Ninth Circuit's text of the a "Injunction" is not a a a a a a a a a a a a a a a a a a a a a a a
  [2] src≈ a. This Court's decision in Heller instructs that, in reviewing a challenge to an arms regulation …
       pred: The Court's text of the a "rement Clauses a "rement Clauses a "rement"
  [3] src≈ Federal law illustrates some of the types of regula- tions that the Second Amendment permit s. Congress has prohibited …
       pred: The Court's text of the a a a a a a a a a a a a a a a a a a a a a a a a a a a a 
  [4] src≈ New York has long sought to protect public safety by requiring a person to establish "proper cause" to obtain …
       pred: The Ninth Circuit's text of the a "Injunction" is not a a a a a a a a a

### Small preset

Runs **`notebook_scratch_small_full/best.pt`**.


In [4]:
run_inference_for_checkpoint("small", CKPT_SMALL)



=== dev · scratch small (/Users/naataaniitsosie/repos/cs474/runs/notebook_scratch_small_full/best.pt) ===


[dev-small] 100/2319 rows  14.3s wall  (~7.01 rows/s)
[dev-small] 200/2319 rows  26.8s wall  (~7.47 rows/s)
[dev-small] 300/2319 rows  35.1s wall  (~8.54 rows/s)
[dev-small] 400/2319 rows  49.9s wall  (~8.01 rows/s)
[dev-small] 500/2319 rows  60.3s wall  (~8.29 rows/s)
[dev-small] 600/2319 rows  72.8s wall  (~8.24 rows/s)
[dev-small] 700/2319 rows  83.6s wall  (~8.37 rows/s)
[dev-small] 800/2319 rows  94.7s wall  (~8.45 rows/s)
[dev-small] 900/2319 rows  109.1s wall  (~8.25 rows/s)
[dev-small] 1000/2319 rows  124.9s wall  (~8.01 rows/s)
[dev-small] 1100/2319 rows  141.9s wall  (~7.75 rows/s)
[dev-small] 1200/2319 rows  155.8s wall  (~7.70 rows/s)
[dev-small] 1300/2319 rows  172.6s wall  (~7.53 rows/s)
[dev-small] 1400/2319 rows  186.4s wall  (~7.51 rows/s)
[dev-small] 1500/2319 rows  199.7s wall  (~7.51 rows/s)
[dev-small] 1600/2319 rows  214.1s wall  (~7.47 rows/s)
[dev-small] 1700/2319 rows  228.6s wall  (~7.44 rows/s)
[dev-small] 1800/2319 rows  243.9s wall  (~7.38 rows/s)
[dev-smal

Saved → artifacts/inference_runs/dev_scratch_small.json  eval_n=2319
corpus: {'rouge1_f': 0.1429, 'rougeL_f': 0.1289, 'token_f1_macro': 0.1311, 'chrf_corpus': 0.3854}

Per-row log (source ≈ first 24 tok; pred = full string):


Token indices sequence length is longer than the specified maximum sequence length for this model (1132 > 512). Running this sequence through the model will result in indexing errors


  [0] src≈ Safety The Second Amendment states : "A well regulated Militia, being necessary to the security of …
       pred: The Court should not remand by the First Amendment rights of the First Amendment rights
  [1] src≈ a. Firea rms serve valuable purposes, including ena- bling self - …
       pred: The Court Should Not Apply the Constitutional Right to the First Amendments to the First Amendment the First Amendment
  [2] src≈ a. This Court's decision in Heller instructs that, in reviewing a challenge to an arms regulation …
       pred: The Court should not remand to the Court's precedents
  [3] src≈ Federal law illustrates some of the types of regula- tions that the Second Amendment permit s. Congress has prohibited …
       pred: The Court should not remand to the remand to the remand to the remand to the remand by the remand by the a non-based on the a non-based on the a non-based on the a non-repayment limit
  [4] src≈ New York has long sought to protect public safety by requi

### Medium preset

Runs **`notebook_scratch_medium_full/best.pt`**.


In [5]:
run_inference_for_checkpoint("medium", CKPT_MEDIUM)



=== dev · scratch medium (/Users/naataaniitsosie/repos/cs474/runs/notebook_scratch_medium_full/best.pt) ===


[dev-medium] 100/2319 rows  31.6s wall  (~3.16 rows/s)
[dev-medium] 200/2319 rows  62.0s wall  (~3.23 rows/s)
[dev-medium] 300/2319 rows  100.6s wall  (~2.98 rows/s)
[dev-medium] 400/2319 rows  145.0s wall  (~2.76 rows/s)
[dev-medium] 500/2319 rows  181.8s wall  (~2.75 rows/s)
[dev-medium] 600/2319 rows  218.6s wall  (~2.74 rows/s)
[dev-medium] 700/2319 rows  255.7s wall  (~2.74 rows/s)
[dev-medium] 800/2319 rows  284.2s wall  (~2.81 rows/s)
[dev-medium] 900/2319 rows  312.3s wall  (~2.88 rows/s)
[dev-medium] 1000/2319 rows  341.1s wall  (~2.93 rows/s)
[dev-medium] 1100/2319 rows  374.7s wall  (~2.94 rows/s)
[dev-medium] 1200/2319 rows  404.9s wall  (~2.96 rows/s)
[dev-medium] 1300/2319 rows  440.9s wall  (~2.95 rows/s)
[dev-medium] 1400/2319 rows  475.5s wall  (~2.94 rows/s)
[dev-medium] 1500/2319 rows  507.1s wall  (~2.96 rows/s)
[dev-medium] 1600/2319 rows  536.7s wall  (~2.98 rows/s)
[dev-medium] 1700/2319 rows  568.2s wall  (~2.99 rows/s)
[dev-medium] 1800/2319 rows  591.4s wall  

Saved → artifacts/inference_runs/dev_scratch_medium.json  eval_n=2319
corpus: {'rouge1_f': 0.0854, 'rougeL_f': 0.0771, 'token_f1_macro': 0.0731, 'chrf_corpus': 0.1906}

Per-row log (source ≈ first 24 tok; pred = full string):


Token indices sequence length is longer than the specified maximum sequence length for this model (1132 > 512). Running this sequence through the model will result in indexing errors


  [0] src≈ Safety The Second Amendment states : "A well regulated Militia, being necessary to the security of …
       pred: The Court Should Not Appeals's The Government's Decisions The Government's The Government's The Government's The Government's Requirements Clauses The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government's The Government
  [1] src≈ a. Firea rms serve valuable purposes, including ena- bling self - …
       pred: The Court's Decisions a State Courts a State Interests a State Interests a State Interests a State Courts a State Interests a State Interests a Clear of the a State Court's a "Contrac""s"s"s"sssssssssssssssssssssssssssss
  [2] src≈ a. This Court's

### Saved runs (optional)

Lists JSON files under **`artifacts/inference_runs/`** (newest first).

In [7]:
from briefme.inference_persist import list_saved_scratch_runs

paths = list_saved_scratch_runs(REPO_ROOT)
for p in paths[:20]:
    print(p.relative_to(REPO_ROOT))

artifacts/inference_runs/dev_scratch_medium.json
artifacts/inference_runs/dev_scratch_small.json
artifacts/inference_runs/dev_scratch_tiny.json
